# 02 Dataset Preparation

??????????????????

???? GitHub ??????????????????????????


## Source: about_data.ipynb


In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import time

base_url = "https://www.gushiwen.cn"
list_url = "https://www.gushiwen.cn/gushi/tangshi.aspx"
headers = {"User-Agent": "Mozilla/5.0"}

def get_poem_links():
    response = requests.get(list_url, headers=headers)
    response.encoding = 'utf-8'
    soup = BeautifulSoup(response.text, "html.parser")
    links = []
    # 根据页面结构查找所有链接，通常诗链接格式为 "/shiwenvXXXX.aspx"
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if href.startswith("/shiwenv"):
            links.append(base_url + href)
    return list(set(links))  # 去重

def get_poem_data(poem_url):
    response = requests.get(poem_url, headers=headers)
    response.encoding = 'utf-8'
    soup = BeautifulSoup(response.text, "html.parser")
    
    # 提取标题
    title_tag = soup.find("h1")
    title = title_tag.text.strip() if title_tag else ""
    
    # 提取作者和朝代信息，通常在 div class="source" 中
    source = soup.find("div", class_="source")
    dynasty = ""
    author = ""
    if source:
        a_tags = source.find_all("a")
        if a_tags:
            dynasty = a_tags[0].text.strip()  # 通常第一个是朝代
            author = a_tags[-1].text.strip()   # 最后一个是作者
    # 提取正文内容
    content_div = soup.find("div", class_="contson")
    content = content_div.get_text(separator="\n").strip() if content_div else ""
    
    # 提取译文，如果有的话，通常在 div class="contyishang" 或类似标签内
    translation_div = soup.find("div", class_="contyishang")
    translation = translation_div.get_text(separator="\n").strip() if translation_div else ""
    
    return {
        "title": title,
        "dynasty": dynasty,
        "author": author,
        "content": content,
        "translation": translation
    }

# 获取所有诗的链接
poem_links = get_poem_links()
print("找到诗链接数量：", len(poem_links))

all_poems = []
for link in poem_links:
    try:
        data = get_poem_data(link)
        all_poems.append(data)
        print("爬取：", data["title"])
        time.sleep(1)  # 为了不对服务器造成压力，设置延时
    except Exception as e:
        print("爬取出错：", link, e)

# 保存为 JSON 文件
with open("tangshi_poems.json", "w", encoding="utf-8") as f:
    json.dump(all_poems, f, ensure_ascii=False, indent=2)

print("爬取完成，数据已保存。")


In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import time

# 基础URL
base_url = "https://www.gushiwen.cn"
# 唐诗三百首目录页URL
list_url = f"{base_url}/gushi/tangshi.aspx"
# 请求头，模拟浏览器访问
headers = {"User-Agent": "Mozilla/5.0"}

def get_poem_links():
    """
    获取唐诗三百首中每首诗的链接
    """
    response = requests.get(list_url, headers=headers)
    response.encoding = 'utf-8'
    soup = BeautifulSoup(response.text, "html.parser")
    links = []
    # 查找所有诗歌链接
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if href.startswith("/shiwenv"):
            links.append(base_url + href)
    # 去重处理
    return list(set(links))

def get_poem_data(poem_url):
    """
    获取每首诗的详细信息：标题、朝代、作者、正文、译文
    """
    response = requests.get(poem_url, headers=headers)
    response.encoding = 'utf-8'
    soup = BeautifulSoup(response.text, "html.parser")
    
    # 提取标题
    title_tag = soup.find("h1")
    title = title_tag.text.strip() if title_tag else ""
    
    # 提取作者和朝代信息
    source = soup.find("p", class_="source")
    dynasty = ""
    author = ""
    if source:
        a_tags = source.find_all("a")
        if len(a_tags) >= 2:
            dynasty = a_tags[0].text.strip()
            author = a_tags[1].text.strip()
    
    # 提取正文内容
    content_div = soup.find("div", class_="contson")
    content = content_div.get_text(separator="\n").strip() if content_div else ""
    
    # 提取译文
    translation = ""
    for div in soup.find_all("div", class_="contyishang"):
        # 查找包含“译文”的部分
        if div.find_previous_sibling("h2") and "译文" in div.find_previous_sibling("h2").text:
            translation = div.get_text(separator="\n").strip()
            break
    
    return {
        "title": title,
        "dynasty": dynasty,
        "author": author,
        "content": content,
        "translation": translation
    }

def main():
    # 获取所有诗的链接
    poem_links = get_poem_links()
    print(f"共找到 {len(poem_links)} 首诗的链接。")

    all_poems = []
    for index, link in enumerate(poem_links, 1):
        try:
            data = get_poem_data(link)
            all_poems.append(data)
            print(f"({index}/{len(poem_links)}) 爬取：{data['title']}")
            # 为避免过于频繁的请求，增加延时
            time.sleep(1)
        except Exception as e:
            print(f"爬取出错：{link}，错误信息：{e}")

    # 保存为 JSON 文件
    with open("tangshi_poems.json", "w", encoding="utf-8") as f:
        json.dump(all_poems, f, ensure_ascii=False, indent=2)

    print("爬取完成，数据已保存为 tangshi_poems.json。")

if __name__ == "__main__":
    main()


In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import time
import re

# 基础URL
base_url = "https://www.gushiwen.cn"
list_url = f"{base_url}/gushi/tangshi.aspx"
headers = {"User-Agent": "Mozilla/5.0"}

def get_poem_links():
    """
    获取唐诗三百首中每首诗的链接
    """
    response = requests.get(list_url, headers=headers)
    response.encoding = 'utf-8'
    soup = BeautifulSoup(response.text, "html.parser")
    links = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if href.startswith("/shiwenv"):
            links.append(base_url + href)
    return list(set(links))

def get_poem_data(poem_url):
    """
    获取每首诗的详细信息：标题、朝代、作者、正文、译文
    """
    response = requests.get(poem_url, headers=headers)
    response.encoding = 'utf-8'
    soup = BeautifulSoup(response.text, "html.parser")

    # 提取标题
    title_tag = soup.find("h1")
    title = title_tag.text.strip() if title_tag else ""

    # 提取朝代和作者信息
    source = soup.find("p", class_="source")
    dynasty = ""
    author = ""
    if source:
        a_tags = source.find_all("a")
        if len(a_tags) >= 2:
            dynasty = a_tags[1].text.strip()  # 朝代（在后）
            author = a_tags[0].text.strip()  # 作者（在前）

    # 去掉朝代的括号
    dynasty = re.sub(r"[（）]", "", dynasty)

    # 提取正文内容
    content_div = soup.find("div", class_="contson")
    content = content_div.get_text(separator="\n").strip() if content_div else ""

    # 提取译文
    translation = ""
    yizhu_div = soup.find("div", class_="yizhu")
    if yizhu_div:
        for span in yizhu_div.find_all("span", class_=""):
            if "译文" in span.text:
                translation_tag = span.find_next_sibling("p")
                if translation_tag:
                    translation = translation_tag.get_text(separator="\n").strip()
                break  # 找到后立即退出循环

    return {
        "title": title,
        "dynasty": dynasty,
        "author": author,
        "content": content,
        "translation": translation
    }

def main():
    poem_links = get_poem_links()
    print(f"共找到 {len(poem_links)} 首诗的链接。")

    all_poems = []
    for index, link in enumerate(poem_links, 1):
        try:
            data = get_poem_data(link)
            all_poems.append(data)
            print(f"({index}/{len(poem_links)}) 爬取：{data['title']}")
            time.sleep(1)  # 避免过快访问被封
        except Exception as e:
            print(f"爬取出错：{link}，错误信息：{e}")

    # 保存为 JSON 文件
    with open("tangshi_poems.json", "w", encoding="utf-8") as f:
        json.dump(all_poems, f, ensure_ascii=False, indent=2)

    print("爬取完成，数据已保存为 tangshi_poems.json。")

if __name__ == "__main__":
    main()


In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import time
import re

# 基础URL
base_url = "https://www.gushiwen.cn"
list_url = f"{base_url}/gushi/tangshi.aspx"
headers = {"User-Agent": "Mozilla/5.0"}

def get_poem_links():
    """
    获取唐诗三百首中每首诗的链接
    """
    response = requests.get(list_url, headers=headers)
    response.encoding = 'utf-8'
    soup = BeautifulSoup(response.text, "html.parser")
    links = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if href.startswith("/shiwenv"):
            links.append(base_url + href)
    return list(set(links))

def get_poem_data(poem_url):
    """
    获取每首诗的详细信息：标题、朝代、作者、正文、译文
    """
    response = requests.get(poem_url, headers=headers)
    response.encoding = 'utf-8'
    soup = BeautifulSoup(response.text, "html.parser")

    # 提取标题
    title_tag = soup.find("h1")
    title = title_tag.text.strip() if title_tag else ""

    # 提取朝代和作者信息
    source = soup.find("p", class_="source")
    dynasty = ""
    author = ""
    if source:
        a_tags = source.find_all("a")
        if len(a_tags) >= 2:
            dynasty = a_tags[1].text.strip()  # 朝代（在后）
            author = a_tags[0].text.strip()  # 作者（在前）

    # 去掉朝代的括号
    dynasty = re.sub(r"[〔〕]", "", dynasty)

    # 提取正文内容
    content_div = soup.find("div", class_="contson")
    content = content_div.get_text(separator="\n").strip() if content_div else ""

    # **使用最初的翻译爬取方法**
    translation = ""
    translation_div = soup.find("div", class_="contyishang")
    if translation_div:
        translation = translation_div.get_text(separator="\n").strip()

    return {
        "title": title,
        "dynasty": dynasty,
        "author": author,
        "content": content,
        "translation": translation
    }

def main():
    poem_links = get_poem_links()
    print(f"共找到 {len(poem_links)} 首诗的链接。")

    all_poems = []
    for index, link in enumerate(poem_links, 1):
        try:
            data = get_poem_data(link)
            all_poems.append(data)
            print(f"({index}/{len(poem_links)}) 爬取：{data['title']} - 翻译：{'有' if data['translation'] else '无'}")
            time.sleep(1)  # 避免访问过快
        except Exception as e:
            print(f"爬取出错：{link}，错误信息：{e}")

    # 保存为 JSON 文件
    with open("tangshi_poems.json", "w", encoding="utf-8") as f:
        json.dump(all_poems, f, ensure_ascii=False, indent=2)

    print("爬取完成，数据已保存为 tangshi_poems.json。")

if __name__ == "__main__":
    main()


In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import time
import re

# 基础URL
base_url = "https://www.gushiwen.cn"
list_url = f"{base_url}/gushi/songsan.aspx"
headers = {"User-Agent": "Mozilla/5.0"}

def get_poem_links():
    """
    获取宋词三百首中每首诗的链接
    """
    response = requests.get(list_url, headers=headers)
    response.encoding = 'utf-8'
    soup = BeautifulSoup(response.text, "html.parser")
    links = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if href.startswith("/shiwenv"):
            links.append(base_url + href)
    return list(set(links))

def get_poem_data(poem_url):
    """
    获取每首诗的详细信息：标题、朝代、作者、正文、译文
    """
    response = requests.get(poem_url, headers=headers)
    response.encoding = 'utf-8'
    soup = BeautifulSoup(response.text, "html.parser")

    # 提取标题
    title_tag = soup.find("h1")
    title = title_tag.text.strip() if title_tag else ""

    # 提取朝代和作者信息
    source = soup.find("p", class_="source")
    dynasty = ""
    author = ""
    if source:
        a_tags = source.find_all("a")
        if len(a_tags) >= 2:
            dynasty = a_tags[1].text.strip()  # 朝代（在后）
            author = a_tags[0].text.strip()  # 作者（在前）

    # 去掉朝代的括号
    dynasty = re.sub(r"[〔〕]", "", dynasty)

    # 提取正文内容
    content_div = soup.find("div", class_="contson")
    content = content_div.get_text(separator="\n").strip() if content_div else ""

    # **使用最初的翻译爬取方法**
    translation = ""
    translation_div = soup.find("div", class_="contyishang")
    if translation_div:
        translation = translation_div.get_text(separator="\n").strip()

    return {
        "title": title,
        "dynasty": dynasty,
        "author": author,
        "content": content,
        "translation": translation
    }

def main():
    poem_links = get_poem_links()
    print(f"共找到 {len(poem_links)} 首诗的链接。")

    all_poems = []
    for index, link in enumerate(poem_links, 1):
        try:
            data = get_poem_data(link)
            all_poems.append(data)
            print(f"({index}/{len(poem_links)}) 爬取：{data['title']} - 翻译：{'有' if data['translation'] else '无'}")
            time.sleep(1)  # 避免访问过快
        except Exception as e:
            print(f"爬取出错：{link}，错误信息：{e}")

    # 保存为 JSON 文件
    with open("songci_poems.json", "w", encoding="utf-8") as f:
        json.dump(all_poems, f, ensure_ascii=False, indent=2)

    print("爬取完成，数据已保存为 songci_poems.json。")

if __name__ == "__main__":
    main()


In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import time
import re

# 基础URL
base_url = "https://www.gushiwen.cn"
list_url = f"{base_url}/gushi/songsan.aspx"
headers = {"User-Agent": "Mozilla/5.0"}

def get_poem_links():
    """
    获取宋词三百首中每首诗的链接
    """
    response = requests.get(list_url, headers=headers)
    response.encoding = 'utf-8'
    soup = BeautifulSoup(response.text, "html.parser")
    links = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if href.startswith("/shiwenv"):
            links.append(base_url + href)
    return list(set(links))

def get_poem_data(poem_url):
    """
    获取每首诗的详细信息：标题、朝代、作者、正文、译文
    """
    response = requests.get(poem_url, headers=headers)
    response.encoding = 'utf-8'
    soup = BeautifulSoup(response.text, "html.parser")

    # 提取标题
    title_tag = soup.find("h1")
    title = title_tag.text.strip() if title_tag else ""

    # 提取朝代和作者信息
    source = soup.find("p", class_="source")
    dynasty = ""
    author = ""
    if source:
        a_tags = source.find_all("a")
        if len(a_tags) >= 2:
            dynasty = a_tags[1].text.strip()  # 朝代（在后）
            author = a_tags[0].text.strip()  # 作者（在前）

    # 去掉朝代的括号
    dynasty = re.sub(r"[〔〕]", "", dynasty)

    # 提取正文内容
    content_div = soup.find("div", class_="contson")
    content = content_div.get_text(separator="\n").strip() if content_div else ""

    # **使用最初的翻译爬取方法**
    translation = ""
    translation_div = soup.find("div", class_="contyishang")
    if translation_div:
        translation = translation_div.get_text(separator="\n").strip()

    return {
        "title": title,
        "dynasty": dynasty,
        "author": author,
        "content": content,
        "translation": translation
    }

def main():
    poem_links = get_poem_links()
    print(f"共找到 {len(poem_links)} 首诗的链接。")

    all_poems = []
    for index, link in enumerate(poem_links, 1):
        try:
            data = get_poem_data(link)
            all_poems.append(data)
            print(f"({index}/{len(poem_links)}) 爬取：{data['title']} - 翻译：{'有' if data['translation'] else '无'}")
            time.sleep(1)  # 避免访问过快
        except Exception as e:
            print(f"爬取出错：{link}，错误信息：{e}")

    # 保存为 JSON 文件
    with open("songci_poems.json", "w", encoding="utf-8") as f:
        json.dump(all_poems, f, ensure_ascii=False, indent=2)

    print("爬取完成，数据已保存为 songci_poems.json。")

if __name__ == "__main__":
    main()


In [ ]:
# 2. 导入相关模块
import json
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
import torch

In [ ]:
# 3. 加载唐诗数据
# 假设数据文件为 "tangshi_poems.json"，格式为一个列表，每个元素为字典
with open("tangshi_poems.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [ ]:
# 4. 数据预处理
# 为每个样本在 "content" 前加上控制标记 “[唐诗风格]”
def add_control_token(sample):
    # 只使用原诗内容作为训练文本
    sample["text"] = "[唐诗风格] " + sample["content"]
    return sample

# 转换为 Hugging Face 的 Dataset 格式
dataset = Dataset.from_list(data)
dataset = dataset.map(add_control_token)

In [ ]:
# 5. 加载 Qwen 模型和 Tokenizer
model_name = "Qwen/Qwen-Chat-7B"  # 请确保该模型在 Hugging Face Hub 可用，或者根据实际名称进行调整
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

In [ ]:
# 6. Tokenization
def tokenize_function(example):
    return tokenizer(example["text"], truncation=True, max_length=128)

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=dataset.column_names)

In [ ]:
# 7. 划分训练集和验证集（80%/20%）
split_dataset = tokenized_dataset.train_test_split(test_size=0.2)

In [ ]:
# 8. 设置训练参数
training_args = TrainingArguments(
    output_dir="./tangshi_style_model",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    evaluation_strategy="steps",
    eval_steps=500,
    save_steps=500,
    logging_steps=100,
    learning_rate=5e-5,
    weight_decay=0.01,
    save_total_limit=2,
    fp16=True if torch.cuda.is_available() else False,
)


In [ ]:
# 9. 初始化 Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split_dataset["train"],
    eval_dataset=split_dataset["test"],
)

In [ ]:
# 10. 开始训练
trainer.train()

In [ ]:
# 11. 保存微调后的模型和分词器
model.save_pretrained("./tangshi_style_model")
tokenizer.save_pretrained("./tangshi_style_model")

In [ ]:
# 12. 定义生成函数进行风格测试
def generate_style_text(input_text, max_length=100):
    # 加上控制码，使输入符合训练风格
    prompt = "[唐诗风格] " + input_text
    input_ids = tokenizer.encode(prompt, return_tensors="pt")
    if torch.cuda.is_available():
        input_ids = input_ids.to("cuda")
        model.to("cuda")
    output_ids = model.generate(input_ids, max_length=max_length, num_beams=5, no_repeat_ngram_size=2)
    output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return output_text

In [ ]:
# 测试生成效果
test_input = "床前明月光，疑是地上霜。"
print("生成结果：")
print(generate_style_text(test_input))

In [ ]:
#开一个新的,下面这个是关于数据清洗的
#规则是：删除“展开阅读全文”的记录
#在对每条记录进行处理前，会遍历检查 title、dynasty、author、content 和 translation 中是否包含 “展开阅读全文”，若存在，则直接跳过该记录，不进行后续清洗。
#content 分句
#先删除所有括号（全角和半角）内的内容，再使用中文句号、叹号、冒号和换行符进行分句，返回句子列表。
#translation 清洗
#删除前缀“译文及注释\n\n\n\n\n\n\n译文\n”，再分句，遇到包含“注释”的句子时停止，返回前面的句子列表。
#删除含英文记录
#使用 contains_english 函数检查五项中是否含有英文字符，若发现则跳过该记录。

In [ ]:
import re
import json

def clean_content(text):
    """
    对 content 字段进行清洗：
    1. 删除全角括号（……）和半角括号 (...) 内的内容；
    2. 按句号、叹号、冒号和换行符分句，去除空白句子；
    3. 将分句后的内容合并成一个连续字符串。
    """
    # 删除全角括号内的内容
    text = re.sub(r"（[^）]*）", "", text)
    # 删除半角括号内的内容
    text = re.sub(r"\([^)]*\)", "", text)
    
    # 按中文标点分句
    sentences = re.split(r'[。！？：\n]', text)
    # 去除空白句子
    sentences = [s.strip() for s in sentences if s.strip()]
    return "".join(sentences)

def clean_translation(text):
    """
    对 translation 字段进行清洗：
    1. 使用正则删除前缀 "译文及注释\n\n\n\n\n\n\n译文\n"（允许换行和空白）；
    2. 如果存在“注释”，则删除“注释”及其之后的所有内容；
    3. 按句号、叹号、冒号和换行符分句，去除空白句子；
    4. 将分句后的内容合并成一个连续字符串。
    """
    # 删除前缀
    prefix_pattern = r"^译文及注释\s*\n+\s*译文\s*\n+"
    text = re.sub(prefix_pattern, "", text)
    
    # 如果存在“注释”，则只保留“注释”之前的内容
    if "注释" in text:
        text = text.split("注释")[0]
    
    # 分句处理
    sentences = re.split(r'[。！？：\n]', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    return "".join(sentences)

def contains_english(value):
    """
    判断传入的字符串或字符串列表中是否包含英文字母
    """
    if isinstance(value, str):
        return bool(re.search(r'[A-Za-z]', value))
    elif isinstance(value, list):
        for item in value:
            if re.search(r'[A-Za-z]', item):
                return True
        return False
    return False

def main():
    input_file = "tangshi_poems.json"
    output_file = "tangshi_poems_clean.json"
    
    # 读取原始数据
    with open(input_file, "r", encoding="utf-8") as f:
        poems = json.load(f)
    
    original_count = len(poems)
    cleaned_poems = []
    
    for poem in poems:
        # 对 content 和 translation 先进行清洗处理
        if "content" in poem:
            poem["content"] = clean_content(poem["content"])
        if "translation" in poem:
            poem["translation"] = clean_translation(poem["translation"])
        
        # 清洗后判断是否存在“展开阅读全文”，若存在则跳过该记录
        skip = False
        for field in ["title", "dynasty", "author", "content", "translation"]:
            if field in poem and re.search("展开阅读全文", poem[field]):
                skip = True
                break
        if skip:
            continue
        
        # 检查 title、dynasty、author、content、translation 五项中是否含有英文字符
        for field in ["title", "dynasty", "author", "content", "translation"]:
            if field in poem and contains_english(poem[field]):
                skip = True
                break
        if skip:
            continue
        
        cleaned_poems.append(poem)
    
    cleaned_count = len(cleaned_poems)
    
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(cleaned_poems, f, ensure_ascii=False, indent=4)
    
    print(f"原始数据数量: {original_count}")
    print(f"清洗后数据数量: {cleaned_count}")

if __name__ == "__main__":
    main()


In [ ]:
#以上是关于唐诗的数据清洗，下列是关于宋词的

In [ ]:
import re
import json

def clean_content(text):
    """
    对 content 字段进行清洗：
    1. 删除全角括号（……）和半角括号 (...) 内的内容；
    2. 按句号、叹号、冒号和换行符分句，去除空白句子；
    3. 将分句后的内容合并成一个连续字符串。
    """
    # 删除全角括号内的内容
    text = re.sub(r"（[^）]*）", "", text)
    # 删除半角括号内的内容
    text = re.sub(r"\([^)]*\)", "", text)
    
    # 按中文标点分句
    sentences = re.split(r'[。！？：\n]', text)
    # 去除空白句子
    sentences = [s.strip() for s in sentences if s.strip()]
    return "".join(sentences)

def clean_translation(text):
    """
    对 translation 字段进行清洗：
    1. 使用正则删除前缀 "译文及注释\n\n\n\n\n\n\n译文\n"（允许换行和空白）；
    2. 如果存在“注释”，则删除“注释”及其之后的所有内容；
    3. 按句号、叹号、冒号和换行符分句，去除空白句子；
    4. 将分句后的内容合并成一个连续字符串。
    """
    # 删除前缀
    prefix_pattern = r"^译文及注释\s*\n+\s*译文\s*\n+"
    text = re.sub(prefix_pattern, "", text)
    
    # 如果存在“注释”，则只保留“注释”之前的内容
    if "注释" in text:
        text = text.split("注释")[0]
    
    # 分句处理
    sentences = re.split(r'[。！？：\n]', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    return "".join(sentences)

def contains_english(value):
    """
    判断传入的字符串或字符串列表中是否包含英文字母
    """
    if isinstance(value, str):
        return bool(re.search(r'[A-Za-z]', value))
    elif isinstance(value, list):
        for item in value:
            if re.search(r'[A-Za-z]', item):
                return True
        return False
    return False

def main():
    input_file = "songci_poems.json"
    output_file = "songci_poems_clean.json"
    
    # 读取原始数据
    with open(input_file, "r", encoding="utf-8") as f:
        poems = json.load(f)
    
    original_count = len(poems)
    cleaned_poems = []
    
    for poem in poems:
        # 对 content 和 translation 先进行清洗处理
        if "content" in poem:
            poem["content"] = clean_content(poem["content"])
        if "translation" in poem:
            poem["translation"] = clean_translation(poem["translation"])
        
        # 清洗后判断是否存在“展开阅读全文”，若存在则跳过该记录
        skip = False
        for field in ["title", "dynasty", "author", "content", "translation"]:
            if field in poem and re.search("展开阅读全文", poem[field]):
                skip = True
                break
        if skip:
            continue
        
        # 检查 title、dynasty、author、content、translation 五项中是否含有英文字符
        for field in ["title", "dynasty", "author", "content", "translation"]:
            if field in poem and contains_english(poem[field]):
                skip = True
                break
        if skip:
            continue
        
        cleaned_poems.append(poem)
    
    cleaned_count = len(cleaned_poems)
    
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(cleaned_poems, f, ensure_ascii=False, indent=4)
    
    print(f"原始数据数量: {original_count}")
    print(f"清洗后数据数量: {cleaned_count}")

if __name__ == "__main__":
    main()


In [ ]:
#数据处理完了 2025.4.10